In [ ]:
import fitz
import re
from collections import Counter
from langchain_core.documents import Document

def production_pdf_loader(pdf_path, header_pct=0.08, footer_pct=0.08):
    pdf = fitz.open(pdf_path)
    raw_pages = []
    print(pdf.metadata)

    # Step 1 — Extract with coordinate cropping
    for page_num, page in enumerate(pdf):
        h = page.rect.height
        w = page.rect.width
        rect = fitz.Rect(0, h * header_pct, w, h * (1 - footer_pct))
        text = page.get_text("text", clip=rect)
        raw_pages.append(text)
    pdf.close()
    
    # Step 2 — Remove remaining repeated lines
    all_lines = []
    for p in raw_pages:
        all_lines.extend([l.strip() for l in p.split('\n') if l.strip()])
    counts = Counter(all_lines)
    repeated = {l for l, c in counts.items() if c / len(raw_pages) >= 0.5}

    # print(repeated)
    
    # Step 3 — Clean each page
    docs = []
    for i, text in enumerate(raw_pages):
        lines = text.split('\n')
        cleaned = []
        for line in lines:
            stripped = line.strip()
            if stripped in repeated:
                continue
            if re.match(r'^\s*\d+\s*$', stripped):  # page numbers
                continue
            cleaned.append(line)
        
        # Fix word-per-line artifacts
        text = '\n'.join(cleaned)
        text = re.sub(r'  +', ' ', text)
        
        docs.append(Document(
            page_content=text.strip(),
            metadata={"page": i + 1, "source": pdf_path}
        ))
    
    return docs

# Usage
pages = production_pdf_loader("../policies/Holiday-Policy.pdf")

In [ ]:
pages = [page.page_content for page in pages]

In [ ]:
pages[1:]